## Ingestão de dados - pedidos

#### 1.Criar sessão spark e carregar configurações

In [1]:
import sys
sys.path.append("/app")

from utils import create_spark_session, load_config, save_table

spark = create_spark_session("pedidos")
config = load_config()

db = config["sqlserver"]
tables = config["tables"]
jdbc_url = f"jdbc:sqlserver://{db['host']}:{db['port']};databaseName={db['database']}"

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/04/09 03:26:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/09 03:26:15 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


#### 2.Executar consulta SQL

In [2]:
tabela_nome = "pedidos"
query = ""

if tabela_nome in [table["name"] for table in tables]:

    query = f"""
        SELECT * FROM {tabela_nome}
    """

#### 3.Armazenar dados na camada bronze

In [3]:
df = spark.read \
     .format("jdbc") \
     .option("url", jdbc_url) \
     .option("query", query) \
     .option("user", db["user"]) \
     .option("password", db["password"]) \
     .option("driver", db["driver"]) \
     .option("encrypt", "true") \
     .option("trustServerCertificate", "true") \
     .load()

table_config = next(
    table for table in tables
    if table["name"] == tabela_nome
)

save_table(
    df=df,
    table_config=table_config,
    config=config,
    spark=spark,
    layer="bronze"
)


Processando tabela: pedidos
path: data/bronze/pedidos


modo incremental: MERGE
usando watermark


ultimo valor de data_atualizado: 2026-04-06 16:29:25.620000


sem novos dados
